# ch07 Bonus 03：Gradio 对话交互界面

> 对照官方 `ch07/06_user_interface`

## 一句话

给指令微调模型套一个 **Gradio 对话 UI**：用户在浏览器输入「指令 + 输入」，模型实时生成响应。

## 原理

把 `generate(instruction, input) → response` 函数用 `gr.Interface` 包装。和 ch06 的情感分类 UI 区别：这里是**开放式生成**，而非单标签分类。

In [ ]:
import torch
import torch.nn.functional as F
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M

# 快速训练 demo 模型（同主线）
TRAIN_DATA = [
    {"instruction": "识别情感", "input": "今天天气真好", "output": " 正面"},
    {"instruction": "识别情感", "input": "太让人失望了", "output": " 负面"},
    {"instruction": "翻译成英文", "input": "你好", "output": " Hello"},
    {"instruction": "翻译成英文", "input": "谢谢", "output": " Thank you"},
    {"instruction": "回答问题", "input": "法国首都是哪", "output": " 巴黎"},
] * 5

def format_prompt(instr, inp):
    return f"### Instruction:\n{instr}\n### Input:\n{inp}\n### Response:\n"

tok = tiktoken.get_encoding("gpt2")
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 64})
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(123)
model = GPTModel(cfg)
for p in model.parameters(): p.requires_grad = False
for p in model.trf_blocks[-1].parameters(): p.requires_grad = True
for p in model.final_norm.parameters(): p.requires_grad = True
for p in model.out_head.parameters(): p.requires_grad = True
model.to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)

def collate(batch):
    xs, ys = [], []
    for e in batch:
        p = tok.encode(format_prompt(e["instruction"], e["input"]))
        r = tok.encode(e["output"])
        ids = (p+r)[:64]; t = (ids[1:]+[tok.eot_token])[:64]
        for i in range(min(len(p), len(t))): t[i] = -100
        ids = ids + [50256]*(64-len(ids)); t = t + [-100]*(64-len(t))
        xs.append(ids); ys.append(t)
    return torch.tensor(xs), torch.tensor(ys)

model.train()
for _ in range(10):
    for i in range(0, len(TRAIN_DATA), 4):
        x, y = collate(TRAIN_DATA[i:i+4])
        opt.zero_grad()
        F.cross_entropy(model(x.to(device)).flatten(0,1), y.to(device).flatten(),
                       ignore_index=-100).backward()
        opt.step()
print("✓ demo 模型已训练")

In [ ]:
def answer(instruction, user_input):
    """Gradio 包装的函数：输入指令+内容，返回模型响应。"""
    model.eval()
    prompt = format_prompt(instruction, user_input)
    idx = torch.tensor([tok.encode(prompt)]).to(device)
    with torch.no_grad():
        for _ in range(10):
            logits = model(idx[:, -cfg["context_length"]:])[:, -1, :]
            nid = logits.argmax(-1, keepdim=True)
            idx = torch.cat([idx, nid], dim=1)
            if nid.item() == tok.eot_token: break
    resp = tok.decode(idx[0].tolist()).split("### Response:\n")[-1].strip()
    return resp.replace("<|endoftext|>", "").strip() or "(无输出)"

# 测试函数
for instr, inp in [("识别情感", "今天天气真好"), ("翻译成英文", "你好")]:
    print(f"[{instr}] {inp} → {answer(instr, inp)!r}")

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=answer,
    inputs=[
        gr.Textbox(label="指令", value="翻译成英文", placeholder="如：翻译成英文 / 识别情感"),
        gr.Textbox(label="输入内容", value="你好", placeholder="要处理的内容"),
    ],
    outputs=gr.Textbox(label="模型响应", lines=3),
    title="指令微调对话 demo",
    description="输入指令和内容，模型生成响应（demo 用未训练小模型，加载预训练权重后更准）。",
)

print("Gradio 界面已定义。")
print("启动方式（终端）: demo.launch() → http://127.0.0.1:7860")
print("\n这里不实际 launch（会阻塞 notebook）；取消注释下行启动：")
# demo.launch()